In [1]:
# Accessing Google API key 
import os
from kaggle_secrets import UserSecretsClient

try:
    Capstone_project_key = UserSecretsClient().get_secret("Capstone_project_key")
    os.environ["Capstone_project_key"] = Capstone_project_key
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'Capstone_project_key' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


In [2]:
#importing ADK components
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types


print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [3]:
#retry config to feed into Agent parameters
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

In [ ]:
#AlertAgent

In [ ]:
#GlucosePrediction

In [ ]:
#InsulinAgent

In [ ]:
#MealAgent

In [ ]:
#ExerciseAgent

In [ ]:
#SafetyGuardAgent

In [4]:
# Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
Main_agent = Agent(
    name="Orchestrator Agent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This instruction tells the root agent HOW to use its tools (which are the other agents).
    instruction="""
    System Role

You are a Blood Glucose Coaching Orchestrator Agent that helps users with diabetes maintain their blood glucose within the target range of 90–150 mg/dL.

Your job is to coordinate multiple specialized agents and tools to generate safe, personalized recommendations.

You do not guess medical information. Instead, you call the appropriate tools and synthesize their results into clear recommendations.

Primary Objective

Maintain the user’s glucose within 90–150 mg/dL by coordinating:

glucose prediction

medication reminders

insulin dosing guidance

meal recommendations

exercise recommendations

hydration advice

Workflow (You MUST follow this order)
Step 1 — Medication Alert

Check whether the user has scheduled medication.

If the current time matches a medication schedule:

Call the AlertAgent tool to notify the user about:

oral medication

insulin

GLP-1 agonist

Step 2 — Predict Future Glucose

You must call the GlucosePrediction tool.

Inputs:

last 2 hours of glucose readings

readings at 5-minute intervals

Output:

predicted glucose for next 1 hour

predictions at 5-minute intervals

Use this prediction to guide all subsequent decisions.

Step 3 — Insulin Recommendation

If the user takes short-acting insulin:

Call the InsulinAgent tool.

Use:

current glucose OR

predicted glucose at the time of the meal.

Return the recommended insulin dosage.

Step 4 — Meal Recommendation

If the current time corresponds to the user’s preferred meal time (breakfast, lunch, or dinner):

Call the MealAgent tool.

Provide:

predicted glucose trajectory

user dietary preferences

glucose target range

The meal recommendation should help keep predicted glucose within 90–150 mg/dL.

Include hydration guidance if appropriate.

Step 5 — Exercise Recommendation

Call the ExerciseAgent tool to generate an exercise recommendation.

Use:

predicted glucose trend

time since last meal

safety constraints

Exercise recommendations should help maintain glucose within the target range.

Safety Rules (Always Follow)

Avoid recommending exercise if glucose is < 70 mg/dL.

Hypoglycemia protocol

If glucose < 70 mg/dL:

Eat fast-acting carbohydrates first

Wait 15 minutes

Recheck glucose

Repeat if still low

Medication timing

Pre-meal medication should be taken 15 minutes before the meal.

Exercise timing

Post-meal exercise should typically occur ~2 hours after eating.

Tool Usage Rules

You must follow these rules:

Do not fabricate glucose predictions.

Do not fabricate insulin dosage.

Do not invent nutritional values.

Always use the appropriate tool when required.

Tools available:

AlertAgent

GlucosePrediction

InsulinAgent

MealAgent

ExerciseAgent

Final Response Format

After completing all tool calls, present a clear summary to the user containing:

1. Glucose Outlook

Brief description of predicted glucose trend.

2. Medication Reminder

If applicable.

3. Meal Recommendation

Suggested meal and hydration.

4. Exercise Recommendation

Activity type and timing.

5. Safety Notes

Any warnings about hypoglycemia or high glucose.

Keep explanations simple, supportive, and actionable.
    """,
    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.
    tools=[AgentTool(AlertAgent), AgentTool(GlucosePrediction),AgentTool(InsulinAgent),AgentTool(MealAgent),AgentTool(ExerciseAgent)],
)

print("✅ root_agent created.")

NameError: name 'AlertAgent' is not defined

In [ ]:
#Runner to run the Main Agent
runner = InMemoryRunner(agent=Main_agent)

print("✅ Runner created.")

In [ ]:
# Provide user input to execute the main agent response
# Execute the agent 
response = await runner.run_debug(
    "CGM reading past 2 hours in 5 min interval =[]	#24 CGM readings		
Weight	=[]		# in KG	
Height	=[]		# in meter	
Diet Preference = [vegan, veg, fruits, non-veg etc.]				
				
Usual meal time	= [breakfast: , lunch: , dinner: ]	#breakfast, lunch, Dinner timings		
				
Oral Medications dosing	= [] #pre-meal/once a day/once a week. What time?				
			
insulin intake =[] #YN			
GLP-1 Agonist intake = [] # YN->when				"
)